In [1]:
import mlflow
from typing import cast
from mlflow.entities import Expectation, Trace, AssessmentSource , AssessmentSourceType
MLFLOW_TRACKING_URI = "http://100.113.186.28:5000"
EXPERIMENT_NAME = "vds-agent-validation"
EXPERIMENT_ID = "6"
TRACE_FILTER = 'trace.text LIKE "%I want%"'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/6', creation_time=1775830300571, experiment_id='6', last_update_time=1775830300571, lifecycle_stage='active', name='vds-agent-validation', tags={'mlflow.latestOnlineScoring.trace.checkpoint': '{"timestamp_ms": '
                                                '1776493059184, "trace_id": '
                                                'null}'}, workspace='default'>

In [2]:
import json
with open('/home/tinhanhnguyen/Desktop/HK8/Capstone/CAPSTONE_PROJECT/videodeepsearch/local/mlflow_eval_records.json', mode='r') as f:
    eval_record_dict = json.load(f)

In [3]:
traces = cast(
    list[Trace],
    mlflow.search_traces(
        locations=[EXPERIMENT_ID],
        filter_string=TRACE_FILTER,
        return_type='list'
    )
)

In [4]:
def get_record_based_on_trace(
    data_df: list[dict],
    trace: Trace
): 
    """
    Get the corresponding record based on trace's session id
    """
    trace_input_preview = cast(str, trace.info.request_preview).strip('"').strip("'").replace('\\n', '')
    
    filter_data_record = next(
        filter(
            lambda x: x['inputs']['user_demand'].replace('\n', '') == trace_input_preview, data_df
        )
    )
    
    return filter_data_record

In [7]:
for trace in traces:
    expectations = trace.search_assessments(type='expectation')
    
    for expectation in expectations:
        assessment_id = expectation.assessment_id
        mlflow.delete_assessment(
            trace_id=trace.info.trace_id,
            assessment_id=assessment_id
        )
    # print(expectations[0])
    # break

In [ ]:
# for trace in traces:
#     eval_record = get_record_based_on_trace(eval_record_dict, trace)
    
#     expected_response = eval_record['expectations']['expected_response']
    
#     trace_id = trace.info.trace_id
#     name = 'Answer_Expectations'
    
#     mlflow.log_expectation(
#         trace_id=trace_id,
#         name="expected_response",
#         value=expected_response,
#         source=AssessmentSource(
#             source_type=AssessmentSourceType.HUMAN
#         ),
#     )